# Adaptive SEM Inspection — one Colab workflow

Run these cells in order. You do **not** need to open the numbered notebooks. Source, data, checkpoints, and results persist in Google Drive, so a Colab reset can resume at a completed command boundary.

Before running: select `Runtime` → `Change runtime type` → `T4 GPU`, and upload the current project ZIP to MyDrive. The next cell creates a temporary Python 3.12 environment because current Colab base runtimes can be Python 3.13 while this locked GPU stack requires Python 3.12. If NTU's MIIC transfer is unreliable, use a verified Drive folder and set `MIIC_MANUAL_BUNDLE` below.

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile

drive.mount('/content/drive')

# Change only this path if you uploaded the ZIP to another Drive folder.
# Use new paths once so an older Python-3.13 launcher cannot be reused.
ZIP_ON_DRIVE = Path('/content/drive/MyDrive/adaptive-sem-inspection-v0.1.6-colab-python312.zip')
PROJECT = Path('/content/drive/MyDrive/adaptive-sem-inspection-v0.1.6')
RUNTIME_ROOT = Path('/content/drive/MyDrive/adaptive-sem-inspection-runtime-v0.1.6')

# Leave as None for automatic download. If the official MIIC transfer is unreliable,
# put the five verified files in a Drive folder and set its path here. The launcher
# also accepts Anomaly_train.rar.part-000 and .part-001 in place of that archive.
MIIC_MANUAL_BUNDLE = None
# MIIC_MANUAL_BUNDLE = Path('/content/drive/MyDrive/MIIC_verified_files')

if not ZIP_ON_DRIVE.is_file():
    raise FileNotFoundError(f'Upload the project ZIP first: {ZIP_ON_DRIVE}')
if not (PROJECT / 'scripts' / 'run_colab_pipeline.py').is_file():
    PROJECT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_ON_DRIVE) as archive:
        archive.extractall(PROJECT)
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
print('Project:', PROJECT)
print('Persistent runtime:', RUNTIME_ROOT)

In [ ]:
import json
import os
import subprocess
import sys

# The environment is deliberately local and temporary. The source/data/results
# remain in Drive and survive a runtime reset.
COLAB_VENV = Path('/content/adaptive-sem-python312')
COLAB_PYTHON = COLAB_VENV / 'bin' / 'python'

if not COLAB_PYTHON.is_file():
    bootstrap_env = os.environ.copy()
    bootstrap_env.setdefault('UV_CACHE_DIR', '/content/uv-cache')
    bootstrap_env.setdefault('UV_PYTHON_INSTALL_DIR', '/content/uv-python')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', 'uv'], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'python', 'install', '3.12'], env=bootstrap_env, check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', '--managed-python', '--seed', '--allow-existing', str(COLAB_VENV)], env=bootstrap_env, check=True)

# `--seed` gives the runner its own pip. Do not call ensurepip after uv seeds it.

python_version = subprocess.check_output([str(COLAB_PYTHON), '--version'], text=True).strip()
if not python_version.startswith('Python 3.12.'):
    raise RuntimeError(f'Expected the local Python 3.12 runner, found: {python_version}')
print('Stage runner:', COLAB_PYTHON)
print(python_version)

SEVEN_ZIP_URL = 'https://github.com/ip7z/7zip/releases/download/26.04/7z2604-linux-x64.tar.xz'
SEVEN_ZIP_ROOT = Path('/content/7zip-26.04')
SEVEN_ZIP_ARCHIVE = Path('/content/7z2604-linux-x64.tar.xz')
SEVEN_ZIP_BIN = Path('/content/modern-7zip-bin')
if not (SEVEN_ZIP_ROOT / '7zz').is_file():
    subprocess.run(['wget', '--quiet', '--show-progress', '-O', str(SEVEN_ZIP_ARCHIVE), SEVEN_ZIP_URL], check=True)
    SEVEN_ZIP_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(['tar', '-xJf', str(SEVEN_ZIP_ARCHIVE), '-C', str(SEVEN_ZIP_ROOT)], check=True)
    (SEVEN_ZIP_ROOT / '7zz').chmod(0o755)
SEVEN_ZIP_BIN.mkdir(parents=True, exist_ok=True)
shim = SEVEN_ZIP_BIN / '7z'
if shim.exists() or shim.is_symlink():
    shim.unlink()
shim.symlink_to(SEVEN_ZIP_ROOT / '7zz')
os.environ['PATH'] = f'{SEVEN_ZIP_BIN}:{os.environ["PATH"]}'
subprocess.run(['7z', 'i'], stdout=subprocess.DEVNULL, check=True)
print('Modern 7-Zip is ready.')

def _show_last_failure():
    state_path = RUNTIME_ROOT / 'artifacts' / 'colab_pipeline_state.json'
    if not state_path.is_file():
        return
    state = json.loads(state_path.read_text())
    failed = [(key, value) for key, value in state.get('commands', {}).items() if value.get('status') == 'failed']
    if not failed:
        return
    key, record = failed[-1]
    print('Failed command:', record['label'])
    print('Error:', record['error'])
    log_path = RUNTIME_ROOT / 'logs' / 'colab_pipeline' / f'{key}.log'
    if log_path.is_file():
        print('\n'.join(log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-100:]))


def run_stage(stage, *extra):
    environment = os.environ.copy()
    environment['SEMSPECT_RUNTIME_ROOT'] = str(RUNTIME_ROOT)
    environment['MPLCONFIGDIR'] = str(RUNTIME_ROOT / 'cache' / 'matplotlib')
    command = [str(COLAB_PYTHON), 'scripts/run_colab_pipeline.py', '--stage', stage, '--runtime-root', str(RUNTIME_ROOT), *extra]
    try:
        subprocess.run(command, cwd=PROJECT, env=environment, check=True)
    except subprocess.CalledProcessError as error:
        print(f'\nStage {stage!r} stopped with exit code {error.returncode}.')
        _show_last_failure()
        print('Full logs:', RUNTIME_ROOT / 'logs' / 'colab_pipeline')
        print('After resolving the printed error, rerun this same stage; completed commands are retained.')
        raise

## Development

Read and accept the MIIC CC BY-NC 4.0 license before running this cell. It downloads and verifies data, builds the detector/router, calibrates policies, and stops before the held-out final test. If the official MIIC transfer was unreliable, first set `MIIC_MANUAL_BUNDLE` in the setup cell to the Drive folder containing the verified files; the launcher verifies them and rebuilds a sharded training archive automatically.

In [ ]:
development_args = ['--install', '--accept-miic-license']
if MIIC_MANUAL_BUNDLE is not None:
    development_args.extend(['--miic-manual-bundle', str(MIIC_MANUAL_BUNDLE)])
run_stage('development', *development_args)

## Sealed final benchmark

Run only after development succeeds. The launcher creates the required source commit, seals the protocol, extracts final-test features once, and writes the core report.

In [ ]:
run_stage('final', '--install', '--initialize-git')

## Exploratory post-lock analyses

This runs Carinthia, defect-size/prevalence, stress robustness, ablations, failure galleries, and refreshes the report.

In [ ]:
run_stage('postlock', '--install')
print(RUNTIME_ROOT / 'runs' / 'core_v1_final' / 'report.md')